# Non SkLearn Models

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from pyexpat import features
from scipy import stats
import os

# extra imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

#### Import preprocessed data

In [6]:
PROCESSED_DATA_PATH = 'data/processed'

# 1 - non-scaled

X_train_non_scaled  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_non_scaled.csv')
X_val_non_scaled    = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_non_scaled.csv')
X_test_non_scaled   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_non_scaled.csv')

# 2 - robust scaled

X_train_scaled_robust = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_robust_scaled.csv')
X_val_scaled_robust   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_robust_scaled.csv')
X_test_scaled_robust  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_robust_scaled.csv')

# 3 - min-max scaled

X_train_scaled_minmax = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_minmax_scaled.csv')
X_val_scaled_minmax   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_minmax_scaled.csv')
X_test_scaled_minmax  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_minmax_scaled.csv')

# 4 - standard scaled
X_train_scaled_standard = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_standard_scaled.csv')
X_val_scaled_standard   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_standard_scaled.csv')
X_test_scaled_standard  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_standard_scaled.csv')

# y
y_train = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_train.csv')
y_val   = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_val.csv')


X_train_full = pd.concat([X_train_non_scaled, X_val_non_scaled])
y_train_full = pd.concat([y_train, y_val])

print("\nAll versions loaded.")


All versions loaded.


In [7]:
'''
training.py
'''
import json
from datetime import datetime
import os
import shutil

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.utils.parallel import Parallel, delayed
import joblib
import wandb
import matplotlib.pyplot as plt


def refit_full(model, X_full: pd.DataFrame, y_full: pd.Series) -> None:
    """Refit best model on train+val combined before submission."""

    print("Refitting best model on full training data (train + val)...")
    model.fit(X_full, y_full)


def compute_metrics(y_real: pd.Series, y_pred: pd.Series) -> list[float]:
    accuracy = accuracy_score(y_real,y_pred)
    f1_macro =f1_score(y_real,y_pred, average='macro')
    precision_macro =precision_score(y_real,y_pred,  average='macro')
    recall_macro =recall_score(y_real,y_pred,  average='macro')
    classif_report = classification_report(y_real, y_pred)
    return [accuracy, f1_macro, precision_macro, recall_macro, classif_report]


def evaluate(base_name: str, model, X_val: pd.DataFrame, y_val: pd.Series,  best_params: dict|None = None) -> None:
    '''
    Evaluates the best model on the validation data and defines the experiment.
    '''
    print("Evaluating best model on unseen validation data...")

    y_pred = model.predict(X_val)

    define_experiment(base_name, compute_metrics(y_val, y_pred), y_val, y_pred)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics import confusion_matrix

def plot_spatial_confusion(y_true, y_pred, base_name="Model"):
    # 1. Define physical coordinates (radius, angle_in_degrees) based on your image
    polar_coords = {
        0: (2, 0),   1: (2, 45),  2: (2, 90),  3: (2, 135), 4: (2, 180),
        5: (5, 0),   6: (5, 45),  7: (5, 90),  8: (5, 135), 9: (5, 180)
    }

    # Convert polar to Cartesian (x, y) coordinates for plotting
    coords = {}
    for label, (r, theta) in polar_coords.items():
        rad = np.radians(theta)
        coords[label] = (r * np.cos(rad), r * np.sin(rad))

    # Calculate standard confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(10))

    fig, ax = plt.subplots(figsize=(12, 8))

    # 2. Draw the Tracking Unit (centered at origin, pointing outward)
    tracking_unit = patches.Rectangle((-1.5, -1.5), 3, 1.5, color='black', zorder=5)
    ax.add_patch(tracking_unit)
    ax.text(0, -0.75, 'Tracking\nunit', color='white', ha='center', va='center', weight='bold', zorder=6)

    # 3. Draw the background dashed guidelines
    for label in [5, 6, 7, 8, 9]:
        x, y = coords[label]
        ax.plot([0, x], [0, y], color='black', linestyle='--', alpha=0.6, zorder=1)

    # 4. Plot the position nodes (0 through 9)
    for label, (x, y) in coords.items():
        ax.plot(x, y, 'o', markersize=25, color='white', markeredgecolor='black', zorder=4)
        ax.text(x, y, str(label), ha='center', va='center', fontsize=12, zorder=5)

    # 5. Draw the confusion arrows
    # Find the maximum off-diagonal value so we can scale the arrow thickness
    off_diag_mask = ~np.eye(cm.shape[0], dtype=bool)
    max_conf = np.max(cm[off_diag_mask]) if np.any(cm[off_diag_mask]) else 1

    for i in range(10):
        for j in range(10):
            # Only plot off-diagonal elements (errors) where count > 0
            if i != j and cm[i, j] > 0:
                count = cm[i, j]
                x1, y1 = coords[i] # True position
                x2, y2 = coords[j] # Predicted position (where it was mistakenly placed)

                # Scale arrow thickness (linewidth) and opacity (alpha) based on error frequency
                lw = max(1, (count / max_conf) * 5)
                alpha = min(0.3 + (count / max_conf) * 0.7, 1.0)

                # Create a curved arrow so bidirectional confusions (i->j and j->i) don't overlap
                arrow = patches.FancyArrowPatch(
                    (x1, y1), (x2, y2),
                    connectionstyle="arc3,rad=0.15",
                    arrowstyle="->,head_length=8,head_width=4",
                    color='red',
                    linewidth=lw,
                    alpha=alpha,
                    shrinkA=15, # Leaves a gap so the arrow doesn't overlap the circle text
                    shrinkB=15,
                    zorder=3
                )
                ax.add_patch(arrow)

    # 6. Formatting to match the physical aspect ratio

    ax.set_aspect('equal')
    ax.set_xlim(-6, 6)
    ax.set_ylim(-2, 6)
    ax.axis('off')
    plt.title(f'Spatial Error Map: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()

    return fig


def define_experiment(base_name: str, metrics: list[float], y_val: pd.Series, y_pred: pd.Series,  best_params: dict|None = None) -> None:

    accuracy, f1_macro, precision_macro, recall_macro, classif_report = metrics
    experiment_results = {
        "model_name": base_name,
        "best_hyperparameters": best_params,
        "validation_metrics": {"accuracy": accuracy, "f1_macro": f1_macro}
    }

    print("Initializing Weights & Biases run...")
    wandb.init(
        project="AA1",
        entity="laura-rebollo-crespo-universitat-polit-cnica-de-catalunya",
        name=f"{base_name}_f1-{f1_macro:.4f}",
        config={"model_name": base_name, "best_params": best_params})

    fig, ax = plt.subplots(figsize=(10, 8))

    disp = ConfusionMatrixDisplay.from_predictions(
        y_val,
        y_pred,
        ax=ax,
        cmap='viridis',
        colorbar=False
    )
    plt.title(f'Confusion Matrix: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()
    spatial = plot_spatial_confusion(y_val, y_pred, base_name)

    # LOG IT TO W&B
    wandb.log({
        "val_accuracy": accuracy,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "classification_report": wandb.Html(f"<pre>{classif_report}</pre>"),

        "scikit_learn_matrix": wandb.Image(fig),
        "spatial_confusion_matrix": wandb.Image(spatial)
    })


    plt.close(fig)
    plt.close(spatial)


def save(base_name:str, model) -> None:
    """
    Saves the best model locally and uploads it to W&B as an artifact, then cleans up the local file.
    """
    models_dir = "outputs/models"
    os.makedirs(models_dir, exist_ok=True)

    # Save the Model locally first so W&B can grab it
    model_filepath = f"{models_dir}/{base_name}.pkl"
    print(f"Saving model locally to {model_filepath}...")
    joblib.dump(model, model_filepath)

    # --- 3. UPLOAD MODEL TO W&B ---
    print("Uploading model to W&B Cloud...")
    model_artifact = wandb.Artifact(
        name=f"{base_name}_model",
        type="model",
        description="Trained  model"
    )
    model_artifact.add_file(model_filepath)
    wandb.log_artifact(model_artifact)

    if os.path.exists(model_filepath):
        os.remove(model_filepath)
        # Only remove directory if it's empty; use shutil.rmtree if cleanup needed
        try:
            os.rmdir(models_dir)
        except OSError:
            pass  # Directory not empty or other error - that's fine
        print("Deleted:", model_filepath)
    else:
        print("File not found:", model_filepath)


def save_submission(y_pred: pd.Series, file_name: str) -> None:
    """
    Generates Kaggle predictions and uploads the CSV to W&B.
    """
    submissions_dir = "outputs/submissions"
    os.makedirs(submissions_dir, exist_ok=True)

    output_path = f"{submissions_dir}/{file_name}.csv"

    print("Generating Kaggle submission...")

    submission_ids = range(len(y_pred))

    submission = pd.DataFrame({
        "ID": submission_ids,
        "POSITION": y_pred.astype(int)
    })

    submission.to_csv(output_path, index=False)
    print(f"Submission saved locally to: {output_path}")

    # UPLOAD CSV TO W&B ---
    print("Uploading Kaggle submission to W&B Cloud...")
    csv_artifact = wandb.Artifact(
        name=f"{file_name}_submission",
        type="predictions"
    )
    csv_artifact.add_file(output_path)
    wandb.log_artifact(csv_artifact)

    # --- 5. CLOSE THE W&B RUN ---
    wandb.finish()



#### Helper functions

In [8]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

## Models

### TabPFN

In [ ]:
%pip install tabpfn-client

In [ ]:
print("--- Training TabPFN ---")
from tabpfn_client import TabPFNClassifier, set_access_token

TABPFN_KEY = ''
# Configure TabPFN
set_access_token(TABPFN_KEY)
model_tabpfn = TabPFNClassifier()

# Fit the model
model_tabpfn.fit(X_train_scaled_robust, y_train.values.ravel())

# Define model metadata
best_params_tabpfn = {"description": "Pre-trained Foundation Model"}

# Use the helper functions from training.py
evaluate("TabPFN", model_tabpfn, X_val_scaled_robust, y_val, best_params=best_params_tabpfn)
save("TabPFN", model_tabpfn)

refit_full(model_tabpfn, X_train_full, y_train_full)
save_submission(model_tabpfn.predict(X_test_scaled_robust), 'TabPFN-robust-scaled')

### TabNet

In [ ]:
!pip install pytorch-tabnet

from pytorch_tabnet.tab_model import TabNetClassifier
import torch
from sklearn.model_selection import GridSearchCV

print("--- Cross-Validating and Training TabNet ---")

# Data preparation (Standard Scaling is preferred for TabNet)
X_t = X_train_scaled_standard.values
y_t = y_train.values.ravel()
X_v = X_val_scaled_standard.values
y_v = y_val.values.ravel()

# Define the base model
tabnet_model = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',
    verbose=0
)

# Define parameter grid for cross-validation
param_grid = {
    'n_d': [8, 16],
    'n_a': [8, 16],
    'optimizer_params': [dict(lr=2e-2), dict(lr=1e-2)]
}

# Initialize Grid Search
grid_search = GridSearchCV(
    tabnet_model,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    verbose=1
)

# Run Cross-Validation
grid_search.fit(
    X_t, y_t,
    eval_set=[(X_v, y_v)],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)

print(f"Best Params: {grid_search.best_params_}")
model_tabnet = grid_search.best_estimator_

# Use helper functions for validation metrics and W&B logging
evaluate("TabNet_CV", model_tabnet, X_v, y_val, best_params=grid_search.best_params_)
save("TabNet_CV", model_tabnet)

# Final refit on full training data (train + val)
refit_full(model_tabnet, X_train_full.values, y_train_full.values.ravel())

# Generate and save submission
y_pred_tabnet = model_tabnet.predict(X_test_scaled_standard.values)
save_submission(y_pred_tabnet, 'TabNet-CV-standard-scaled')

### 1D CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
import numpy as np

print("--- Training 1D CNN with Cross-Validation ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define 1D CNN Architecture
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(CNN1D, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            if isinstance(X, pd.DataFrame) or isinstance(X, np.ndarray):
                X_vals = X.values if hasattr(X, 'values') else X
                X_tensor = torch.FloatTensor(X_vals).unsqueeze(1).to(device)
            else:
                X_tensor = X
            outputs = self.forward(X_tensor)
            return torch.argmax(outputs, dim=1).cpu().numpy()

# Data and CV setup
X_full_cv = X_train_scaled_standard.values
y_full_cv = y_train.values.ravel()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_cv, y_full_cv)):
    print(f"Training Fold {fold+1}/5...")

    X_tr, X_va = X_full_cv[train_idx], X_full_cv[val_idx]
    y_tr, y_va = y_full_cv[train_idx], y_full_cv[val_idx]

    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr).unsqueeze(1).to(device), torch.LongTensor(y_tr).to(device)), batch_size=64, shuffle=True)

    model_cnn = CNN1D(X_full_cv.shape[1], 10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

    for epoch in range(30):
        model_cnn.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model_cnn(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    y_va_pred = model_cnn.predict(X_va)
    acc = accuracy_score(y_va, y_va_pred)
    fold_metrics.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

print(f"Mean CV Accuracy: {np.mean(fold_metrics):.4f}")

evaluate("1D-CNN-CV", model_cnn, X_val_scaled_standard.values, y_val, best_params={"mean_cv_acc": np.mean(fold_metrics)})
save("1D-CNN-CV", model_cnn)

y_pred_cnn = model_cnn.predict(X_test_scaled_standard.values)
save_submission(y_pred_cnn, 'CNN1D-CV-standard-scaled')


### CatBoost

### XgBoost - Distance Aware

In [ ]:
def refit_full_no_es(model, X_full, y_full):
    params = model.get_params()
    params.pop("early_stopping_rounds", None)  # remove ES for final fit
    final_model = xgb.XGBClassifier(**params)
    final_model.fit(X_full, y_full)
    return final_model

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import numpy as np

print("--- Training XGBoost with GPU + Distance-Aware Loss ---")

# 1. Exact coordinates (meters)
coords = {
    0: (-2.000, 0.000), 1: (-1.414, 1.414), 2: (0.000, 2.000), 3: (1.414, 1.414), 4: (2.000, 0.000),
    5: (-5.000, 0.000), 6: (-3.535, 3.535), 7: (0.000, 5.000), 8: (3.535, 3.535), 9: (5.000, 0.000)
}

num_classes = 10
D = np.zeros((num_classes, num_classes), dtype=np.float32)
for i in range(num_classes):
    for j in range(num_classes):
        D[i, j] = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))

alpha = 0.7
class_weights = 1 + alpha * (D.sum(axis=1) - np.diag(D)) / (num_classes - 1)

# 2. Data preparation
X_t = X_train_scaled_standard.values.astype(np.float32)
y_t = y_train.values.ravel()
X_v = X_val_scaled_standard.values.astype(np.float32)
y_v = y_val.values.ravel()

sample_weights = np.array([class_weights[y] for y in y_t])

# 3. Base model - Fixed: early_stopping_rounds goes in the constructor for XGBoost 2.0+
xgb_model = xgb.XGBClassifier(
    n_estimators=1500,
    device='cuda',
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42
)

param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 4. Grid Search
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    verbose=3,
    n_jobs=1
)

# Fit with eval_set for early stopping
grid_search_xgb.fit(
    X_t, y_t,
    sample_weight=sample_weights,
    eval_set=[(X_v, y_v)],
    verbose=False
)

print("Best Params:", grid_search_xgb.best_params_)
model_xgb = grid_search_xgb.best_estimator_

# 5. Evaluation + W&B logging
evaluate("XGBoost-GPU-DistanceAware", model_xgb, X_v, y_v, best_params=grid_search_xgb.best_params_)
save("XGBoost-GPU-DistanceAware", model_xgb)

# 6. Final refit and submission
X_full = X_train_full.values.astype(np.float32)
y_full = y_train_full.values.ravel()
model_xgb = refit_full_no_es(model_xgb, X_full, y_full)

y_pred_xgb = model_xgb.predict(X_test_scaled_standard.values.astype(np.float32))
save_submission(y_pred_xgb, "XGBoost-GPU-DistanceAware")

### XGBoost - Distance aware + penalitza closes label

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import numpy as np

print("--- Training XGBoost with GPU + Distance-Aware Loss ---")

# ---------------------------------------------------------
# 1. Exact coordinates (meters)
# ---------------------------------------------------------
coords = {
    0: (-2.000, 0.000), 1: (-1.414, 1.414), 2: (0.000, 2.000),
    3: (1.414, 1.414), 4: (2.000, 0.000),
    5: (-5.000, 0.000), 6: (-3.535, 3.535), 7: (0.000, 5.000),
    8: (3.535, 3.535), 9: (5.000, 0.000)
}

num_classes = 10

# ---------------------------------------------------------
# 2. Distance matrix (Euclidean)
# ---------------------------------------------------------
D = np.zeros((num_classes, num_classes), dtype=np.float32)
for i in range(num_classes):
    for j in range(num_classes):
        D[i, j] = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))

# ---------------------------------------------------------
# 3. Class-level distance weighting
# ---------------------------------------------------------
alpha = 0.7
class_weights = 1 + alpha * (D.sum(axis=1) - np.diag(D)) / (num_classes - 1)

# ---------------------------------------------------------
# 4. Per-sample distance-aware weighting (nearest confusing class)
# ---------------------------------------------------------
beta = 1.0  # strength of nearest-class penalty

nearest_distances = np.zeros(num_classes)
for c in range(num_classes):
    nearest_distances[c] = np.partition(D[c], 1)[1]  # nearest other class

# ---------------------------------------------------------
# 5. Data preparation
# ---------------------------------------------------------
X_t = X_train_scaled_standard.values.astype(np.float32)
y_t = y_train.values.ravel()

X_v = X_val_scaled_standard.values.astype(np.float32)
y_v = y_val.values.ravel()

# Combined per-sample weights
sample_weights = np.array([
    class_weights[y] * (1 + beta * nearest_distances[y])
    for y in y_t
])

# ---------------------------------------------------------
# 6. Base model (GPU + early stopping)
# ---------------------------------------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=1500,
    device='cuda',
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42
)

# ---------------------------------------------------------
# 7. Parameter grid
# ---------------------------------------------------------
param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# ---------------------------------------------------------
# 8. Grid Search with early stopping
# ---------------------------------------------------------
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    verbose=3,
    n_jobs=1
)

grid_search_xgb.fit(
    X_t, y_t,
    sample_weight=sample_weights,
    eval_set=[(X_v, y_v)],
    verbose=False
)

print("Best Params:", grid_search_xgb.best_params_)
model_xgb = grid_search_xgb.best_estimator_

# ---------------------------------------------------------
# 9. Evaluation + W&B logging
# ---------------------------------------------------------
evaluate(
    "XGBoost-GPU-DistanceAware",
    model_xgb,
    X_v,
    y_v,
    best_params=grid_search_xgb.best_params_
)

save("XGBoost-GPU-DistanceAware", model_xgb)

# ---------------------------------------------------------
# 10. Final refit and submission
# ---------------------------------------------------------
X_full = X_train_full.values.astype(np.float32)
y_full = y_train_full.values.ravel()

model_xgb = refit_full_no_es(model_xgb, X_full, y_full)

y_pred_xgb = model_xgb.predict(
    X_test_scaled_standard.values.astype(np.float32)
)

save_submission(y_pred_xgb, "XGBoost-GPU-DistanceAware")

### AutoGluon

In [5]:
print("\n--- Training AutoGluon ---")

from autogluon.tabular import TabularPredictor

# 1. Data
train_df = X_train_non_scaled.copy()
train_df["position"] = y_train.values.ravel()

# 2. Fit
model_ag = TabularPredictor(
    label="position",
    eval_metric="f1_macro"
).fit(
    train_data=train_df,
    time_limit=6000,
    presets="best_quality",
    dynamic_stacking=False, 
    num_bag_folds=5,
    verbosity=3
)

# 3. Metadata
best_params_ag = {
    "best_model": model_ag.get_model_best_name(),
    "presets": "best_quality"
}

# 4. Evaluate
evaluate(
    "AutoGluon",
    model_ag,
    X_val_non_scaled,
    y_val,
    best_params=best_params_ag
)

save("AutoGluon", model_ag)

# 5. Refit on full data
print("\nRefitting AutoGluon on full training data...")
model_ag.refit_full(model="all", set_best_to_refit=True)

# 6. Predict test set
y_pred_ag = model_ag.predict(X_test_non_scaled)

# 7. Save submission
save_submission(y_pred_ag.values, "AutoGluon-BestQuality")


--- Training AutoGluon ---


c:\Users\laura\Desktop\UNI\Q4\AA1\ML-WiFi-Sensing-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No path specified. Models will be saved in: "AutogluonModels\ag-20260517_093052"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       18.51 GB / 31.50 GB (58.8%)
Disk Space Avail:   688.53 GB / 951.65 GB (72.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Beginning AutoGluon train

[1000]	valid_set's multi_logloss: 0.295116	valid_set's f1_macro: 0.930619
[1000]	valid_set's multi_logloss: 0.316948	valid_set's f1_macro: 0.927713
[1000]	valid_set's multi_logloss: 0.318398	valid_set's f1_macro: 0.928786


	0.9277	 = Validation score   (f1_macro)
	545.05s	 = Training   runtime
	6.73s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 3329.21s of the 5330.10s of remaining time.
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)


[1000]	valid_set's multi_logloss: 0.360029	valid_set's f1_macro: 0.923607


	0.9249	 = Validation score   (f1_macro)
	341.09s	 = Training   runtime
	2.57s	 = Validation runtime
Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 2983.07s of the 4983.96s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=16, gpus=0, mem=0.5/16.6 GB
	0.8915	 = Validation score   (f1_macro)
	11.6s	 = Training   runtime
	1.25s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 2969.67s of the 4970.56s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=16, gpus=0, mem=0.5/16.5 GB
	0.8933	 = Validation score   (f1_macro)
	15.24s	 = Training   runtime
	1.21s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 2952.72s of the 4953.61s of remaining time.
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)
	Ran out of time, early stopping on iteration 1153.
	Ran o

AttributeError: 'TabularPredictor' object has no attribute 'get_model_best_name'

In [6]:

# 3. Metadata
best_model_name = model_ag.leaderboard(silent=True).iloc[0]["model"]

best_params_ag = {
    "best_model": best_model_name,
    "presets": "best_quality"
}


# 4. Evaluate
evaluate(
    "AutoGluon",
    model_ag,
    X_val_non_scaled,
    y_val,
    best_params=best_params_ag
)

save("AutoGluon", model_ag)

# 5. Refit on full data
print("\nRefitting AutoGluon on full training data...")
model_ag.refit_full(model="all", set_best_to_refit=True)

# 6. Predict test set
y_pred_ag = model_ag.predict(X_test_non_scaled)

# 7. Save submission
save_submission(y_pred_ag.values, "AutoGluon-BestQuality")

Evaluating best model on unseen validation data...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\laura\_netrc.


Initializing Weights & Biases run...


wandb: Currently logged in as: laura-rebollo-crespo (laura-rebollo-crespo-universitat-polit-cnica-de-catalunya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Saving model locally to outputs/models/AutoGluon.pkl...
Uploading model to W&B Cloud...


...

Deleted: outputs/models/AutoGluon.pkl

Refitting AutoGluon on full training data...


No improvement since epoch 0: early stopping
	10.98s	 = Training   runtime
Fitting 1 L1 models, fit_strategy="sequential" ...
Fitting model: LightGBMXT_BAG_L1_FULL ...
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.1/17.0 GB
	46.87s	 = Training   runtime
Fitting 1 L1 models, fit_strategy="sequential" ...
Fitting model: LightGBM_BAG_L1_FULL ...
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.1/16.1 GB
	35.75s	 = Training   runtime
Fitting model: RandomForestGini_BAG_L1_FULL | Skipping fit via cloning parent ...
	11.6s	 = Training   runtime
	1.25s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1_FULL | Skipping fit via cloning parent ...
	15.24s	 = Training   runtime
	1.21s	 = Validation runtime
Fitting 1 L1 models, fit_strategy="sequential" ...
Fitting model: CatBoost_BAG_L1_FULL ...
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0
	335.93s	 = Training   runtime
Fitting model: ExtraTreesGini_BAG_L1_FULL | Skipping fit via clo

Generating Kaggle submission...
Submission saved locally to: outputs/submissions/AutoGluon-BestQuality.csv
Uploading Kaggle submission to W&B Cloud...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.95849
val_f1_macro,0.95863
val_precision_macro,0.9594
val_recall_macro,0.95825


In [7]:
model_ag.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,0.953507,f1_macro,20.535328,4492.243858,0.003693,1.417404,3,True,20
1,LightGBM_BAG_L2,0.952402,f1_macro,14.647019,4225.132542,0.809594,264.936232,2,True,13
2,LightGBMXT_BAG_L2,0.950864,f1_macro,14.830560,4209.654814,0.993135,249.458504,2,True,12
3,XGBoost_BAG_L2,0.950702,f1_macro,14.026634,4025.993533,0.189209,65.797223,2,True,19
4,WeightedEnsemble_L2,0.950582,f1_macro,13.023243,1183.382287,0.008028,2.198228,2,True,10
5,ExtraTreesGini_BAG_L2,0.950414,f1_macro,14.675862,3961.793434,0.838437,1.597124,2,True,17
6,ExtraTreesEntr_BAG_L2,0.949851,f1_macro,14.657542,3961.631431,0.820117,1.435121,2,True,18
7,NeuralNetFastAI_BAG_L2,0.949591,f1_macro,14.344552,4112.115248,0.507127,151.918938,2,True,11
8,CatBoost_BAG_L2,0.949002,f1_macro,13.975813,5150.406244,0.138388,1190.209934,2,True,16
9,RandomForestGini_BAG_L2,0.948182,f1_macro,15.603574,3979.975710,1.766149,19.779400,2,True,14


In [9]:
# 3. Get ensemble model
lb = model_ag.leaderboard(silent=True)
ensemble_name = lb[lb['model'].str.contains('WeightedEnsemble')].iloc[0]['model']

best_params_ag = {
    "best_model": ensemble_name,
    "presets": "best_quality"
}

# 4. Evaluate
evaluate(
    "AutoGluon-Ensemble",
    model_ag,
    X_val_non_scaled,
    y_val,
    best_params=best_params_ag
)

save("AutoGluon-Ensemble", model_ag)

# 5. Refit ensemble on full data
print("\nRefitting AutoGluon ensemble on full training data...")
model_ag.refit_full(model=ensemble_name, set_best_to_refit=True)

# 6. Predict test set
# 3. Get ensemble model
lb = model_ag.leaderboard(silent=True)
ensemble_name = lb[lb['model'].str.contains('WeightedEnsemble')].iloc[0]['model']

best_params_ag = {
    "best_model": ensemble_name,
    "presets": "best_quality"
}

# 4. Evaluate
evaluate(
    "AutoGluon",
    model_ag,
    X_val_non_scaled,
    y_val,
    best_params=best_params_ag
)

save("AutoGluon", model_ag)

# 5. Refit ensemble on full data
print("\nRefitting AutoGluon ensemble on full training data...")
model_ag.refit_full(model=ensemble_name, set_best_to_refit=True)

# 6. Predict test set
y_pred_ag = model_ag.predict(X_test_non_scaled)

# 7. Save submission
save_submission(y_pred_ag.values, "AutoGluon-ensemble")



Evaluating best model on unseen validation data...
Initializing Weights & Biases run...


Saving model locally to outputs/models/AutoGluon-Ensemble.pkl...
Uploading model to W&B Cloud...


Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...
	Models trained in this way will have the suffix "_FULL" and have NaN validation score.
	This process is not bound by time_limit, but should take less time than the original `predictor.fit` call.
	To learn more, refer to the `.refit_full` method docstring which explains how "_FULL" models differ from normal models.
Model 'WeightedEnsemble_L3' already has a refit _FULL model: 'WeightedEnsemble_L3_FULL', skipping refit...
Model 'NeuralNetFastAI_BAG_L1' already has a refit _FULL model: 'NeuralNetFastAI_BAG_L1_FULL', skipping refit...
Model 'ExtraTreesGini_BAG_L1' already has a refit _FULL model: 'ExtraTreesGini_BAG_L1_FULL', skipping refit...
Model 'NeuralNetFastAI_BAG_L2' already has a refit _FULL model: 'NeuralNetFastAI_BAG_L2_FULL', skipping refit...
Model 'LightGBM_BAG_L2' already has a refit _FULL model: 'LightGBM_BAG_L2_FULL', skipping refit...
Model 'RandomForestGini_BAG_L2' already

Deleted: outputs/models/AutoGluon-Ensemble.pkl

Refitting AutoGluon ensemble on full training data...
Evaluating best model on unseen validation data...
Initializing Weights & Biases run...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.91932
val_f1_macro,0.91931
val_precision_macro,0.92307
val_recall_macro,0.91839


Saving model locally to outputs/models/AutoGluon.pkl...
Uploading model to W&B Cloud...


Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...
	Models trained in this way will have the suffix "_FULL" and have NaN validation score.
	This process is not bound by time_limit, but should take less time than the original `predictor.fit` call.
	To learn more, refer to the `.refit_full` method docstring which explains how "_FULL" models differ from normal models.
Model 'WeightedEnsemble_L3' already has a refit _FULL model: 'WeightedEnsemble_L3_FULL', skipping refit...
Model 'NeuralNetFastAI_BAG_L1' already has a refit _FULL model: 'NeuralNetFastAI_BAG_L1_FULL', skipping refit...
Model 'ExtraTreesGini_BAG_L1' already has a refit _FULL model: 'ExtraTreesGini_BAG_L1_FULL', skipping refit...
Model 'NeuralNetFastAI_BAG_L2' already has a refit _FULL model: 'NeuralNetFastAI_BAG_L2_FULL', skipping refit...
Model 'LightGBM_BAG_L2' already has a refit _FULL model: 'LightGBM_BAG_L2_FULL', skipping refit...
Model 'RandomForestGini_BAG_L2' already

Deleted: outputs/models/AutoGluon.pkl

Refitting AutoGluon ensemble on full training data...
Generating Kaggle submission...
Submission saved locally to: outputs/submissions/AutoGluon-ensemble.csv
Uploading Kaggle submission to W&B Cloud...


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.91932
val_f1_macro,0.91931
val_precision_macro,0.92307
val_recall_macro,0.91839


## Ensemble

In [1]:
import wandb
import os
import joblib
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.linear_model import LogisticRegression

# Inicia sessió a l'API de W&B
api = wandb.Api()

USER = "laura-rebollo-crespo-universitat-polit-cnica-de-catalunya"
PROJECT = "AA1"

c:\Users\laura\Desktop\UNI\Q4\AA1\ML-WiFi-Sensing-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\laura\_netrc.


In [23]:
def load_wandb_model(model_name, version="v0"):
    print(f"Downloading {model_name}...")
    artifact_path = f"{USER}/{PROJECT}/{model_name}_model:{version}"
    artifact = api.artifact(artifact_path)
    download_dir = artifact.download()
    
    pkl_file = [f for f in os.listdir(download_dir) if f.endswith('.pkl')][0]
    model = joblib.load(os.path.join(download_dir, pkl_file))
    
    return model

models = {}

#models['SVC'] = load_wandb_model("SVC", version="v16")
models['XGBoost'] = load_wandb_model("XGBoost-GPU-DistanceAware", version="v0")
models['MLPClassifier'] = load_wandb_model("MLPClassifier", version="v0")
#models['1D-CNN'] = load_wandb_model("1D-CNN", version='v0')

#models['AutoGluon'] = load_wandb_model("AutoGluon", version="v0")

# from tabpfn_client import ClientOptions, TabPFNClassifier

# client_options = ClientOptions(api_key="EL_TEUTOKEN")

# models['TabPFN'] = TabPFNClassifier(
#     client_options=client_options,
#     device="cpu"
# )


print("Tots els models carregats a la memòria!")

print(models)

wandb:   1 of 1 files downloaded.  


wandb:   1 of 1 files downloaded.  


Tots els models carregats a la memòria!
{'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device='cuda', early_stopping_rounds=50,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1500, n_jobs=None,
              num_parallel_tree=None, ...), 'MLPClassifier': MLPClassifier(early_stopping=True, hidden_layer_sizes=[512, 256, 128],
              max_iter=500, random_state=42)}
